# RawGrad — PyTorch Comparison

This experiment compares RawGrad with PyTorch on the same
forward and backward computations.

The goal is not to replace RawGrad with PyTorch.

Instead, PyTorch is used as an independent reference implementation
to verify that RawGrad produces mathematically consistent:

- forward outputs
- losses
- weight gradients
- bias gradients

The comparison uses a simple dense neural network so that the
underlying computations can be inspected directly.

## Why Compare Against PyTorch?

RawGrad implements neural-network operations directly using NumPy.

PyTorch provides a mature automatic differentiation system.

By running equivalent computations in both systems and comparing
their results, we can verify that RawGrad's manually implemented
forward and backward calculations agree with an independent
deep-learning framework.

In [1]:
import sys
from pathlib import Path

import numpy as np
import torch

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from rawgrad.layers import Dense

In [2]:
np.random.seed(42)
torch.manual_seed(42)

X = np.array([
    [1.0, 2.0, 3.0],
    [2.0, 1.0, 0.5],
    [0.5, 1.5, 2.5],
], dtype=np.float64)

targets = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
], dtype=np.float64)

print("Input shape:", X.shape)
print("Target shape:", targets.shape)

Input shape: (3, 3)
Target shape: (3, 2)


## Shared Parameters

Both implementations will use exactly the same weights and biases.

This is important because we want to compare the mathematical
operations themselves rather than compare independently initialized
models.

In [3]:
weights = np.array([
    [0.10, -0.20],
    [0.30,  0.40],
    [-0.50, 0.60],
], dtype=np.float64)

biases = np.array([
    [0.10, -0.10]
], dtype=np.float64)

print("Weights:")
print(weights)

print("\nBiases:")
print(biases)

Weights:
[[ 0.1 -0.2]
 [ 0.3  0.4]
 [-0.5  0.6]]

Biases:
[[ 0.1 -0.1]]


In [4]:
rawgrad_layer = Dense(3, 2)

rawgrad_layer.weights = weights.copy()
rawgrad_layer.biases = biases.copy()

rawgrad_output = rawgrad_layer.forward(X)

print("RawGrad output:")
print(rawgrad_output)

RawGrad output:
[[-0.7   2.3 ]
 [ 0.35  0.2 ]
 [-0.65  1.9 ]]


In [6]:
torch_X = torch.tensor(X, dtype=torch.float64)

torch_layer = torch.nn.Linear(3, 2, dtype=torch.float64)

with torch.no_grad():
    torch_layer.weight.copy_(torch.tensor(weights.T))
    torch_layer.bias.copy_(torch.tensor(biases.reshape(-1)))

torch_output = torch_layer(torch_X)

print("PyTorch output:")
print(torch_output.detach().numpy())

PyTorch output:
[[-0.7   2.3 ]
 [ 0.35  0.2 ]
 [-0.65  1.9 ]]


In [7]:
forward_match = np.allclose(
    rawgrad_output,
    torch_output.detach().numpy(),
    atol=1e-10,
)

print("Forward outputs match:", forward_match)

Forward outputs match: True


In [8]:
rawgrad_loss = np.mean((rawgrad_output - targets) ** 2)

rawgrad_gradient = (
    2.0 * (rawgrad_output - targets) / targets.size
)

rawgrad_layer.backward(rawgrad_gradient)

print("RawGrad MSE loss:", rawgrad_loss)
print("RawGrad weight gradients:")
print(rawgrad_layer.weights_gradient)

print("RawGrad bias gradients:")
print(rawgrad_layer.biases_gradient)

RawGrad MSE loss: 2.5458333333333334
RawGrad weight gradients:
[[-0.60833333  0.55      ]
 [-1.84166667  2.21666667]
 [-3.01666667  3.75      ]]
RawGrad bias gradients:
[-1.          1.13333333]


In [9]:
torch_targets = torch.tensor(targets, dtype=torch.float64)

torch_loss = torch.mean((torch_output - torch_targets) ** 2)

torch_loss.backward()

print("PyTorch MSE loss:", torch_loss.item())

print("PyTorch weight gradients:")
print(torch_layer.weight.grad.detach().numpy())

print("PyTorch bias gradients:")
print(torch_layer.bias.grad.detach().numpy())

PyTorch MSE loss: 2.5458333333333334
PyTorch weight gradients:
[[-0.60833333 -1.84166667 -3.01666667]
 [ 0.55        2.21666667  3.75      ]]
PyTorch bias gradients:
[-1.          1.13333333]


In [10]:
torch_weight_gradient = torch_layer.weight.grad.detach().numpy().T
torch_bias_gradient = torch_layer.bias.grad.detach().numpy()

loss_match = np.isclose(
    rawgrad_loss,
    torch_loss.item(),
    atol=1e-10,
)

weight_gradient_match = np.allclose(
    rawgrad_layer.weights_gradient,
    torch_weight_gradient,
    atol=1e-10,
)

bias_gradient_match = np.allclose(
    rawgrad_layer.biases_gradient,
    torch_bias_gradient,
    atol=1e-10,
)

print("Loss match:", loss_match)
print("Weight gradient match:", weight_gradient_match)
print("Bias gradient match:", bias_gradient_match)

Loss match: True
Weight gradient match: True
Bias gradient match: True


In [11]:
assert forward_match
assert loss_match
assert weight_gradient_match
assert bias_gradient_match

print("RAWGRAD ↔ PYTORCH COMPARISON: PASS")

RAWGRAD ↔ PYTORCH COMPARISON: PASS
